# 01 – Data check: ECOSTRESS night coverage and a first MODIS vs ECOSTRESS comparison

This notebook answers three questions before any modelling:

1. How many ECOSTRESS night acquisitions cover central Istanbul in 2018–2025?
2. How many of them are actually cloud-free (from the ECOSTRESS cloud mask)?
3. What does one hot summer night look like at 1 km (MODIS) and at 70 m (ECOSTRESS)?

Outputs: `data/ecostress_nights.csv` and `figures/01_modis_vs_ecostress.png`.

## Setup
On Colab this clones the repository and installs the requirements. Locally it uses the repository you are in.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/KeremBasimc/istanbul-night-heat-island"
if "google.colab" in sys.modules or Path("/content").exists():
    ROOT = Path("/content/istanbul-night-heat-island")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"
FIGS = ROOT / "figures"
FIGS.mkdir(exist_ok=True)
print("Repository:", ROOT)

## Earthdata login

ECOSTRESS needs a free NASA Earthdata account. The password is never written in this notebook. Use one of these:

- **Colab (browser):** add `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` under *Secrets* (key icon on the left), or type them when prompted.
- **Colab CLI:** pass them with `--env` from your own shell environment.

In [ ]:
import earthaccess

try:  # Colab Secrets, if available
    from google.colab import userdata
    for key in ("EARTHDATA_USERNAME", "EARTHDATA_PASSWORD"):
        if key not in os.environ:
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                pass
except ImportError:
    pass

strategy = "environment" if os.environ.get("EARTHDATA_USERNAME") else "interactive"
auth = earthaccess.login(strategy=strategy)
print("Earthdata login ok:", auth.authenticated)

## 1. ECOSTRESS night acquisitions over central Istanbul

In [ ]:
import pandas as pd
from nhi import data

START, END = "2018-07-01", "2025-09-30"
granules = data.search_ecostress(START, END)
night = granules[granules.is_night].copy()
print(f"{len(granules)} granules, {len(night)} at night (21:00-05:00 local)")
night.groupby(night.local.dt.year).size().rename("night granules").to_frame().T

## 2. How many nights are cloud-free?
Downloads only the small `cloud` and `water` layers, then measures the share of land pixels in the study box that the ECOSTRESS cloud mask marks clear.

In [ ]:
night = data.add_clear_fraction(night, DATA / "ecostress")
per_night = (night.groupby("night_date")
             .agg(clear_frac=("clear_frac", "max"), granule=("granule", "first"), local=("local", "first"))
             .reset_index())
per_night.to_csv(DATA / "ecostress_nights.csv", index=False)

summer = pd.to_datetime(per_night.night_date).dt.month.isin([6, 7, 8, 9])
for thr in (0.9, 0.7, 0.5):
    ok = per_night.clear_frac >= thr
    print(f"clear land >= {thr:.0%}: {ok.sum():3d} nights ({(ok & summer).sum()} in Jun-Sep)")

## 3. One hot summer night: MODIS (1 km) vs ECOSTRESS (70 m)
Picks the clearest summer night, loads the same night's MODIS Aqua LST and plots both on one colour scale.

In [ ]:
import matplotlib.pyplot as plt

candidates = per_night[summer & (per_night.clear_frac >= 0.9)]
if candidates.empty:
    candidates = per_night[summer].sort_values("clear_frac", ascending=False).head(1)
chosen = candidates.sort_values("clear_frac", ascending=False).iloc[0]
row = night[night.granule == chosen.granule].iloc[0]
print("Night of", chosen.night_date, "| ECOSTRESS local time", row.local, "| clear", round(chosen.clear_frac, 2))

eco_path = data.download([row.url_LST], DATA / "ecostress")[0]
eco = data.load_ecostress_lst(eco_path)
modis = data.load_modis_night(pd.Timestamp(chosen.night_date).date(), "aqua")

vmin, vmax = [float(v) for v in eco.quantile([0.02, 0.98])]
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, da, title in [(axes[0], modis, "MODIS Aqua night LST, 1 km"),
                      (axes[1], eco, f"ECOSTRESS night LST, 70 m ({row.local:%H:%M} local)")]:
    im = da.plot.imshow(ax=ax, cmap="inferno", vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_title(title); ax.set_aspect("equal"); ax.set_xlabel(""); ax.set_ylabel("")
fig.colorbar(im, ax=axes, label="Land surface temperature (°C)", shrink=0.85)
fig.suptitle(f"Central Istanbul, night of {chosen.night_date}")
fig.savefig(FIGS / "01_modis_vs_ecostress.png", dpi=150)
plt.show()

## Next
- `02_benchmark_tsharp.ipynb`: TsHARP on the matched nights and the first RMSE against ECOSTRESS.
- Fill in the access dates in `DATA_MANIFEST.md`.